# 🧩 Notebook 3: Blackjack — Real-world extensions


## 🛠️ Setup

```bash
cd 07-object-oriented-design/blackjack
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What you'll learn

The core game from Notebook 2 works, but it's still a toy. Real casinos (and real software!) have to deal with: *betting*, *different playing styles*, and *splits*. Each extension teaches a classic OO lesson:

1. **Betting** → composition (a `Player` *has a* wallet).
2. **Strategies** → Strategy Pattern (composition over inheritance).
3. **Splits** → revisiting the domain model when requirements grow.


## 1️⃣ Re-usable core (copied from Notebook 2)

So this notebook is self-contained and runnable on its own.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod
import random

class Suit(Enum):
    HEARTS = "♥"; DIAMONDS = "♦"; CLUBS = "♣"; SPADES = "♠"

RANKS = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]

@dataclass(frozen=True)
class Card:
    rank: str
    suit: Suit
    def value(self) -> int:
        if self.rank == "A":           return 11
        if self.rank in ("J","Q","K"): return 10
        return int(self.rank)
    def __repr__(self): return f"{self.rank}{self.suit.value}"

@dataclass
class Deck:
    cards: list[Card] = field(default_factory=list)
    def __post_init__(self):
        if not self.cards:
            self.cards = [Card(r, s) for s in Suit for r in RANKS]
    def shuffle(self):      random.shuffle(self.cards)
    def draw(self) -> Card: return self.cards.pop()

@dataclass
class Hand:
    cards: list[Card] = field(default_factory=list)
    def add(self, c): self.cards.append(c)
    def value(self) -> int:
        total = sum(c.value() for c in self.cards)
        aces  = sum(1 for c in self.cards if c.rank == "A")
        while total > 21 and aces:
            total -= 10; aces -= 1
        return total
    def is_bust(self):       return self.value() > 21
    def is_blackjack(self):  return len(self.cards) == 2 and self.value() == 21
    def __repr__(self):      return f"{self.cards} = {self.value()}"

print("Core classes loaded.")


## 2️⃣ The Strategy Pattern — swap playing styles without subclassing

In Notebook 2, `Dealer` *inherited* from `Player` just to change one method. That works for two kinds of seat. But imagine 5 playing styles (Conservative, Aggressive, BasicStrategyChart, …). We'd have a deep class tree just to vary one decision.

The **Strategy Pattern** fixes this: pull the decision out into its own object, and let `Player` *hold* one. This is the textbook example of *composition over inheritance*.

> 🐛 **The trap nobody warns you about: stateful strategies.**
> One strategy object is *shared across every round of the game*. A first draft
> of `HitOnFirstDecision` below looked like this:
>
> ```python
> def __init__(self): self._hit_once = False          # ← per-object memory
> def wants_hit(self, hand):
>     if self._hit_once or hand.value() >= 19: return False
>     self._hit_once = True
>     return True
> ```
>
> It behaves correctly in round 1 and is silently broken from round 2 onward:
> `_hit_once` is never reset, so the player stands forever. The bug is invisible
> — no exception, just a player who quietly stops playing.
>
> **Rule:** a strategy should be a *pure function of the state it is handed*. If
> it needs history, that history belongs to the caller (the hand, the round, the
> table) and should be passed in — not hoarded inside the strategy.


In [ ]:
class HitStrategy(ABC):
    """Any object that can answer: given this hand, should I hit?"""
    @abstractmethod
    def wants_hit(self, hand: Hand) -> bool: ...

class HitUntil(HitStrategy):
    """Hit while the hand value is below `threshold`."""
    def __init__(self, threshold: int): self.threshold = threshold
    def wants_hit(self, hand): return hand.value() < self.threshold

class NeverHit(HitStrategy):
    def wants_hit(self, hand): return False

class HitOnFirstDecision(HitStrategy):
    """Take exactly one card if under 19, then stand.

    Stateless: "have I already hit?" is *derived* from the hand it is given
    (a fresh hand holds 2 cards), instead of being remembered in the object.
    That is what makes the same instance safe to reuse round after round.
    """
    def wants_hit(self, hand):
        return len(hand.cards) == 2 and hand.value() < 19

class Player:
    def __init__(self, name: str, strategy: HitStrategy):
        self.name = name
        self.hand = Hand()
        self.strategy = strategy
    def wants_hit(self) -> bool:
        return self.strategy.wants_hit(self.hand)

# Dealer is now just a Player with a fixed strategy — no subclass needed!
def make_dealer() -> Player:
    return Player("Dealer", HitUntil(17))

print("Strategies ready.")


## 3️⃣ Betting — `Chips` via composition

A `Player` *has a* wallet. We don't make `Player` a subclass of `Wallet` — that would be silly. This is another composition win.


In [ ]:
class Chips:
    def __init__(self, balance: int = 100):
        self.balance = balance
        self.current_bet = 0
    def place_bet(self, amount: int):
        if amount > self.balance:
            raise ValueError(f"Not enough chips: {self.balance} < {amount}")
        self.balance -= amount
        self.current_bet = amount
    def win(self, multiplier: float = 2.0):
        # Standard win returns 2x the bet (original + equal amount)
        self.balance += int(self.current_bet * multiplier)
        self.current_bet = 0
    def push(self):
        # Tie — get the original bet back
        self.balance += self.current_bet
        self.current_bet = 0
    def lose(self):
        self.current_bet = 0
    def __repr__(self): return f"Chips(balance={self.balance}, bet={self.current_bet})"

class BettingPlayer(Player):
    def __init__(self, name, strategy, chips: Chips):
        super().__init__(name, strategy)
        self.chips = chips

# Quick demo of chip math
c = Chips(100)
c.place_bet(20);         print(c)
c.win(multiplier=2.5);   print(c, "(blackjack pays 3:2)")


## 4️⃣ A table that puts it all together


In [ ]:
class Table:
    """Runs rounds with betting, pluggable strategies, and a dealer."""
    def __init__(self, players: list[BettingPlayer]):
        self.players = players
        self.dealer  = make_dealer()

    def play_round(self, bet: int = 10):
        # Fresh deck every round keeps the example simple
        deck = Deck(); deck.shuffle()
        for p in self.players:
            p.hand = Hand()
            p.chips.place_bet(bet)
        self.dealer.hand = Hand()

        # Initial deal
        for _ in range(2):
            for p in self.players + [self.dealer]:
                p.hand.add(deck.draw())

        # Player turns
        for p in self.players:
            while p.wants_hit() and not p.hand.is_bust():
                p.hand.add(deck.draw())
        # Dealer turn
        while self.dealer.wants_hit() and not self.dealer.hand.is_bust():
            self.dealer.hand.add(deck.draw())

        # Settle. The rule is a pure function; the payout is the side effect.
        d = self.dealer.hand.value()
        print(f"Dealer: {self.dealer.hand}")
        for p in self.players:
            outcome = self.outcome(p.hand, self.dealer.hand)
            PAYOUT[outcome](p.chips)
            print(f"  {p.name} ({p.hand.value()}) {outcome}  {p.chips}")

    @staticmethod
    def outcome(player: Hand, dealer: Hand) -> str:
        """Who won? Note the dealer-blackjack cases the first draft forgot:
        a natural does NOT pay 3:2 when the dealer also has one — it pushes."""
        if player.is_bust():                                    return "BUST"
        if player.is_blackjack() and dealer.is_blackjack():     return "PUSH"
        if player.is_blackjack():                               return "BLACKJACK"
        if dealer.is_blackjack():                               return "LOSE"
        if dealer.is_bust():                                    return "WIN"
        if player.value() >  dealer.value():                    return "WIN"
        if player.value() == dealer.value():                    return "PUSH"
        return "LOSE"


# Outcome -> what happens to the chips. A table, not another if/elif chain:
# adding "SURRENDER" (half the bet back) is one new row.
PAYOUT = {
    "BUST":      lambda chips: chips.lose(),
    "LOSE":      lambda chips: chips.lose(),
    "PUSH":      lambda chips: chips.push(),
    "WIN":       lambda chips: chips.win(2.0),
    "BLACKJACK": lambda chips: chips.win(2.5),   # 3:2 → stake back + 1.5×
}

random.seed(42)
table = Table([
    BettingPlayer("Alice",  HitUntil(17),          Chips(100)),
    BettingPlayer("Bob",    NeverHit(),            Chips(100)),
    BettingPlayer("Carol",  HitOnFirstDecision(),  Chips(100)),
])
for round_no in range(1, 4):
    print(f"\n===== Round {round_no} =====")
    table.play_round(bet=10)


## 🔍 Verify the extensions

The statefulness bug from section 2 is exactly the kind of thing a demo round
will not catch — it only shows up on the *second* round. So we assert it.

In [ ]:
def hand(*ranks):
    return Hand([Card(r, Suit.SPADES) for r in ranks])

# ── Strategies answer only from the hand they are given ────────────────
s = HitUntil(17)
assert s.wants_hit(hand("10", "6")) is True     # 16 → hit
assert s.wants_hit(hand("10", "7")) is False    # 17 → stand
assert NeverHit().wants_hit(hand("2", "3")) is False

# The regression test for the shared-strategy bug: ONE instance, many rounds.
carol = HitOnFirstDecision()
for round_no in range(5):
    fresh = hand("5", "6")                      # a fresh 11 every round
    assert carol.wants_hit(fresh) is True, f"strategy went stale in round {round_no}"
    fresh.add(Card("4", Suit.HEARTS))           # now 3 cards
    assert carol.wants_hit(fresh) is False, "it should take exactly one card"
assert not vars(carol), "a strategy with instance state will break on reuse"

# ── Chips: money in equals money out ───────────────────────────────────
c = Chips(100)
c.place_bet(30)
assert (c.balance, c.current_bet) == (70, 30)   # the stake leaves the balance
c.push()
assert (c.balance, c.current_bet) == (100, 0),  "a push must be a no-op overall"
c.place_bet(30); c.lose()
assert (c.balance, c.current_bet) == (70, 0)
c.place_bet(30); c.win(2.0)
assert c.balance == 100, "an even-money win returns stake + stake"
c.place_bet(40); c.win(2.5)
assert c.balance == 160, "3:2 returns stake + 1.5x stake"
try:
    Chips(10).place_bet(50); raise AssertionError("bet larger than the balance")
except ValueError:
    pass

# ── Settlement, including the cases the first draft got wrong ──────────
o = Table.outcome
assert o(hand("A", "K"), hand("10", "9")) == "BLACKJACK"
assert o(hand("A", "K"), hand("A", "Q"))  == "PUSH", "two naturals must push"
assert o(hand("10", "9"), hand("A", "K")) == "LOSE", "a dealer natural beats 19"
assert o(hand("K", "Q", "5"), hand("K", "Q", "5")) == "BUST", "bust loses first"
assert o(hand("10", "9"), hand("K", "Q", "5")) == "WIN"
assert o(hand("10", "9"), hand("10", "9")) == "PUSH"
assert set(PAYOUT) >= {o(hand("A","K"), hand("2","3"))} , "every outcome needs a payout"

# Every outcome the rule can produce must have a payout row (no silent misses).
produced = {o(hand("A","K"), hand("A","Q")), o(hand("A","K"), hand("2","3")),
            o(hand("K","Q","5"), hand("2","3")), o(hand("10","9"), hand("10","8")),
            o(hand("10","8"), hand("10","9")), o(hand("10","9"), hand("10","9"))}
assert produced <= set(PAYOUT), produced - set(PAYOUT)

# ── A full round moves each player's balance by a legal amount ─────────
random.seed(99)
t = Table([BettingPlayer("A", HitUntil(17), Chips(100)),
           BettingPlayer("B", NeverHit(),   Chips(100))])
t.play_round(bet=10)
for p in t.players:
    assert p.chips.current_bet == 0, "every bet must be settled at the end of a round"
    assert p.chips.balance in (90, 100, 110, 115), p.chips.balance

print("✅ strategies are stateless, chip math balances, settlement is complete")

## 5️⃣ When requirements grow: splits (discussion)

If the first two cards are the same rank (say two 8s), real Blackjack lets you **split** the hand into two hands, each with its own bet. Notice what breaks in our current model:

- `Player` has *one* `hand` — but now a player can have *many*.
- `Chips.current_bet` is a single number — but each split hand needs its own bet.

**How would you redesign?** A common refactor:

```
Player ──has──▶ list[HandAndBet]
                  │
                  └── Hand + the wager on that particular hand
```

This is the **Open/Closed Principle** (`O` in SOLID) in action: the *design* stays open to extension (splits, doubles, insurance) without rewriting the core classes.

> 🧪 **Challenge:** implement `split()` on `BettingPlayer` and extend `Table` to loop over each split hand. Try it — then compare your approach to the discussion above.


## 🧭 Recap — the OO toolkit we used

| Tool | Where we used it | Why it helps |
|---|---|---|
| **Single Responsibility** | Ace rule lives in `Hand` | One reason to change |
| **Polymorphism** | `wants_hit()` on every seat | `Game` treats everyone the same |
| **Strategy Pattern** | `HitStrategy` objects | Add playing styles without new classes |
| **Composition over Inheritance** | `Player` *has a* `Strategy` and `Chips` | Flexible, testable, less code |
| **Encapsulation** | `Chips` owns balance & bet math | Callers can't corrupt state |
| **Open/Closed** | Splits/doubles as future extensions | Grow the game without rewriting it |
